# 03 · Retrieval Ablation — `scale_30K` (30,000 papers)
Dense / BM25 / Hybrid Recall@K. Platform: Kaggle T4

In [ ]:
# ══════════════════════════════════════════════════════════════
# SCALE EXPERIMENT CONFIG  ← only N_PAPERS and SCALE_LABEL change between tiers
# ══════════════════════════════════════════════════════════════
import os, sys, json, random
import numpy as np
import pandas as pd
import torch

SCALE_LABEL   = "scale_30K"        # identifier used in output filenames
N_PAPERS      = 30000              # ← THE ONLY NUMBER THAT CHANGES PER TIER
RANDOM_SEED   = 42               # NEVER change this
CHUNK_SIZE    = 400              # tokens
CHUNK_OVERLAP = 50               # tokens
RRF_K         = 60               # standard RRF parameter
TOP_K_STAGE1  = 50               # candidates sent to reranker
EVAL_K_VALUES = [1, 3, 5, 10, 20]

# ── Hardware (fixed by FIX 1) ─────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT    = "../../1_data"
RESULTS_DIR  = f"../../4_results/{SCALE_LABEL}"
GT_PATH      = f"{DATA_ROOT}/eval/ground_truth.json"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Scale : {SCALE_LABEL} | N = {N_PAPERS:,} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import pickle, numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

# Load indexes
df_chunks = pd.read_parquet(f"{RESULTS_DIR}/chunks.parquet")
texts = df_chunks['text'].tolist()
chunk_ids = df_chunks['chunk_id'].tolist()
cord_uids = df_chunks['cord_uid'].tolist()
id_to_cord = dict(zip(chunk_ids, cord_uids))

with open(f"{RESULTS_DIR}/bm25_index.pkl", "rb") as f:
    bm25 = pickle.load(f)

embeddings = np.load(f"{RESULTS_DIR}/bge_m3_embeddings.npy")
model = SentenceTransformer("BAAI/bge-m3", device=DEVICE)

print(f"Loaded: {len(texts):,} chunks")

In [ ]:
# ── Load ground truth (FIX 3: external, not derived from this run) ──────────
import json

if os.path.exists(GT_PATH):
    with open(GT_PATH) as f:
        GROUND_TRUTH = json.load(f)
    print(f"Ground truth loaded: {len(GROUND_TRUTH)} queries")
else:
    print("WARNING: ground_truth.json not found.")
    print("Run General/03_query_set.ipynb first.")
    print("Proceeding with pseudo-labels from THIS run as interim fallback.")
    GROUND_TRUTH = None

In [ ]:
# ── Retrieval functions ───────────────────────────────────────────────────────
def dense_retrieve(query, k=50):
    q_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [chunk_ids[i] for i in top_idx]

def bm25_retrieve(query, k=50):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [chunk_ids[i] for i in top_idx]

def rrf_fuse(lists, k=RRF_K):
    scores = {}
    for ranked in lists:
        for rank, doc_id in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_retrieve(query, k=50):
    dense_res = dense_retrieve(query, k)
    bm25_res  = bm25_retrieve(query, k)
    return rrf_fuse([dense_res, bm25_res])[:k]

In [ ]:
# ── Recall@K evaluation ───────────────────────────────────────────────────────
from collections import defaultdict

# Load queries
with open(f"../../1_data/eval/queries.json") as f:
    QUERIES = json.load(f)

# Build pseudo ground truth if no manual GT
if GROUND_TRUTH is None:
    print("Building pseudo-labels from hybrid retrieve (INTERIM ONLY)...")
    GROUND_TRUTH = {}
    for q in QUERIES:
        top3_ids = hybrid_retrieve(q, k=3)
        GROUND_TRUTH[q] = [id_to_cord[cid] for cid in top3_ids]

systems = {
    "dense":  dense_retrieve,
    "bm25":   bm25_retrieve,
    "hybrid": hybrid_retrieve,
}

recall_results = {s: {k: [] for k in EVAL_K_VALUES} for s in systems}

for query in QUERIES:
    if query not in GROUND_TRUTH:
        continue
    relevant_cords = set(GROUND_TRUTH[query])
    for sys_name, retriever in systems.items():
        ranked_ids = retriever(query, k=max(EVAL_K_VALUES))
        for k in EVAL_K_VALUES:
            top_k_cords = {id_to_cord.get(cid, "") for cid in ranked_ids[:k]}
            r = len(top_k_cords & relevant_cords) / len(relevant_cords)
            recall_results[sys_name][k].append(r)

# Compute means
recall_means = {}
for sys_name in systems:
    recall_means[sys_name] = {k: np.mean(v) for k, v in recall_results[sys_name].items()}

# Print table
print(f"\nRecall@K — {SCALE_LABEL} ({len(QUERIES)} queries, {len(df_chunks):,} chunks)")
print(f"{'K':>4}  {'Dense':>8}  {'BM25':>8}  {'Hybrid':>8}")
for k in EVAL_K_VALUES:
    d = recall_means['dense'][k]
    b = recall_means['bm25'][k]
    h = recall_means['hybrid'][k]
    winner = "←" if h > max(d,b) else ("" if d > b else "BM25")
    print(f"{k:>4}  {d:>8.3f}  {b:>8.3f}  {h:>8.3f}  {winner}")

In [ ]:
# Save recall@k results
rows = []
for k in EVAL_K_VALUES:
    rows.append({"k": k,
                 "dense":  recall_means['dense'][k],
                 "bm25":   recall_means['bm25'][k],
                 "hybrid": recall_means['hybrid'][k]})
df_recall = pd.DataFrame(rows)
df_recall.to_csv(f"{RESULTS_DIR}/recall_at_k.csv", index=False)
print(f"Saved: {RESULTS_DIR}/recall_at_k.csv")
print(f"BM25 R@1 advantage: {recall_means['bm25'][1] - recall_means['dense'][1]:+.4f}")

In [ ]:
# ── DPR baseline (FIX 4: DEVICE variable defined above) ──────────────────────
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
import torch

print("Loading DPR baseline...")
dpr_model = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
dpr_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
dpr_model = dpr_model.to(DEVICE)  # DEVICE is defined — no crash
dpr_model.eval()

# Encode all chunks with DPR
BATCH = 32
dpr_embs = []
with torch.no_grad():
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i+BATCH]
        enc = dpr_tokenizer(batch, max_length=512, truncation=True,
                            padding=True, return_tensors="pt").to(DEVICE)
        out = dpr_model(**enc).pooler_output.cpu().numpy()
        dpr_embs.append(out)
dpr_embs = np.vstack(dpr_embs)
print(f"DPR embeddings: {dpr_embs.shape}")
np.save(f"{RESULTS_DIR}/dpr_embeddings.npy", dpr_embs)